### Fine-tuning an Emoji Chatbot using OpenAI
This notebook adapts the Chapter 9 example from the book to use the OpenAI API instead of Azure OpenAI. It demonstrates how to prepare training data, validate it, upload it, fine-tune a model, monitor training, and test the fine-tuned model.

In [75]:
# import required packages
import os
import json
from openai import OpenAI

# initialize the OpenAI client
os.environ["OPENAI_API_KEY"] = 'YOUR_KEY_HERE'
client = OpenAI()

In [76]:
# Create a small JSONL file with ten training examples. OpenAI’s fine-tuning API requires a minimum of 10 training examples for gpt-3.5-turbo.
# Each example follows the chat completion format for fine-tuning.
training_data = [
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "I heard the koala habitat was destroyed by the fire."},
        {"role": "assistant", "content": "(sadkoala)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "My car just broke down on the highway!"},
        {"role": "assistant", "content": "(angrycar)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "It's my birthday today!"},
        {"role": "assistant", "content": "(partyface)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "I just got a promotion at work!"},
        {"role": "assistant", "content": "(clappinghands)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "My dog ran away this morning."},
        {"role": "assistant", "content": "(cryingdog)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "I’m feeling really sick today."},
        {"role": "assistant", "content": "(sickface)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "It’s raining outside again."},
        {"role": "assistant", "content": "(raincloud)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "The new movie was amazing!"},
        {"role": "assistant", "content": "(popcornhappy)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "I lost my phone somewhere at the park."},
        {"role": "assistant", "content": "(lostphone)"}
    ]},
    {"messages": [
        {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
        {"role": "user", "content": "I just adopted a kitten!"},
        {"role": "assistant", "content": "(happycat)"}
    ]}
]

# Save the data in the proper .jsonl format for fine-tuning
with open("emoji_chatbot.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + "\n")

print("✅ Updated training file created with 10 examples.")

✅ Updated training file created with 10 examples.


In [77]:
# Basic file validation
def basic_checks(filename):
    with open(filename, "r") as f:
        lines = f.readlines()
    print(f"✅ Found {len(lines)} examples in {filename}")
    print("First example preview:")
    print(lines[0])

def format_checks(filename):
    with open(filename, "r") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
                assert "messages" in obj, "Missing 'messages' key"
            except Exception as e:
                print(f"❌ Format error on line {i}: {e}")
                return
    print("✅ Format checks passed")

# Run checks
basic_checks("emoji_chatbot.jsonl")
format_checks("emoji_chatbot.jsonl")

✅ Found 10 examples in emoji_chatbot.jsonl
First example preview:
{"messages": [{"role": "system", "content": "You're a chatbot that only responds with emojis!"}, {"role": "user", "content": "I heard the koala habitat was destroyed by the fire."}, {"role": "assistant", "content": "(sadkoala)"}]}

✅ Format checks passed


In [78]:
# Upload the JSONL file for fine-tuning
file = client.files.create(
    file=open("emoji_chatbot.jsonl", "rb"),
    purpose="fine-tune"
)

print(f"✅ File uploaded successfully. File ID: {file.id}")

✅ File uploaded successfully. File ID: file-4ArqPoXzRmdJpHf2eCzvTX


In [79]:
# Start fine-tuning using gpt-3.5-turbo as the base model
job = client.fine_tuning.jobs.create(
    training_file=file.id,
    model="gpt-3.5-turbo",
    suffix="emoji-bot"
)

print("🟢 Fine-tuning job started:")
print(job)
print(f"Job ID: {job.id}")

🟢 Fine-tuning job started:
FineTuningJob(id='ftjob-lb9XGzX9EsCEm4qQh7jNxcn4', created_at=1760930057, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-uH1BOAwW6oyEmbcPAKh7NQOZ', result_files=[], seed=638286873, status='validating_files', trained_tokens=None, training_file='file-4ArqPoXzRmdJpHf2eCzvTX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='emoji-bot', usage_metrics=None, shared_with_openai=False, eval_id=None)
Job ID: ftjob-lb9XGzX9EsCEm4qQh7jNxcn4


In [80]:
# Polling to check fine-tuning job status
import time
from IPython.display import clear_output

# Start tracking time
start_time = time.time()

# Get job ID from previous step
JOB_ID = job.id

# Retrieve the job and get its current status
ft_job = client.fine_tuning.jobs.retrieve(JOB_ID)
status = ft_job.status

# Keep polling until the job succeeds or fails
while status not in ['succeeded', 'failed']:
    ft_job = client.fine_tuning.jobs.retrieve(JOB_ID)
    status = ft_job.status

    clear_output(wait=True)  # Refresh notebook output
    print(f"🟡 Status: {status}")
    print(ft_job)  # Optional: shows detailed progress info

    # Calculate and print elapsed time
    elapsed_minutes = int((time.time() - start_time) // 60)
    elapsed_seconds = int((time.time() - start_time) % 60)
    print(f"Elapsed time: {elapsed_minutes} minutes {elapsed_seconds} seconds")

    # Wait 30 seconds before checking again
    time.sleep(30)

# Print final results once finished
if status == 'succeeded':
    print("✅ Fine-tuning job completed successfully!")
    print(f"Fine-tuned model: {ft_job.fine_tuned_model}")
else:
    print("❌ Fine-tuning job failed. Check job details above.")

🟡 Status: succeeded
FineTuningJob(id='ftjob-lb9XGzX9EsCEm4qQh7jNxcn4', created_at=1760930057, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-3.5-turbo-0125:personal:emoji-bot:CSakKtQd', finished_at=1760930470, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=10), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-uH1BOAwW6oyEmbcPAKh7NQOZ', result_files=['file-QSvXUpwCpWQUf2c1wmW8VJ'], seed=638286873, status='succeeded', trained_tokens=3580, training_file='file-4ArqPoXzRmdJpHf2eCzvTX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=10))), user_provided_suffix='emoji-bot', usage_metrics=None, shared_with_openai=False, eval_id=None)
Elapsed time: 19 minutes 43 seconds
✅ Fine-tuning job comp

In [85]:
# List the recent fine-tuning jobs
jobs = client.fine_tuning.jobs.list(limit=5)
for job in jobs.data:
    print(f"Job ID: {job.id}, Status: {job.status}, Fine-tuned model: {job.fine_tuned_model}")

Job ID: ftjob-lb9XGzX9EsCEm4qQh7jNxcn4, Status: succeeded, Fine-tuned model: ft:gpt-3.5-turbo-0125:personal:emoji-bot:CSakKtQd
Job ID: ftjob-iRGLdrydMoKbhcdrJ73wSFsb, Status: succeeded, Fine-tuned model: ft:gpt-3.5-turbo-0125:personal:emoji-bot:CSaeF6v8
Job ID: ftjob-rpfhLwnCNj1TzCjG5d5krhoQ, Status: failed, Fine-tuned model: None
Job ID: ftjob-TxCnRwTdRFl4LQ2ws8gt73C4, Status: failed, Fine-tuned model: None
Job ID: ftjob-GycL38erdWcBfsLh71lOdUlv, Status: failed, Fine-tuned model: None


In [87]:
# Test the fine-tune model
completion = client.chat.completions.create(
  model="ft:gpt-3.5-turbo-0125:personal:emoji-bot:CSakKtQd",
  messages=[
    {"role": "system", "content": "You're a chatbot that only responds with emojis!"},
    {"role": "user", "content": "What the hell is going on?"}
  ],
  max_tokens=50
)

print("💬 Model Response:")
print(completion.choices[0].message.content)

💬 Model Response:
(confusedface)
